In [ ]:
!pip install pandas

In [ ]:
import pandas as pd

# =========================
# 1️⃣ CSV'leri yükle (FOOD)
# =========================
food_df = pd.read_csv(
    "/content/drive/MyDrive/datas/food.csv",
    usecols=["fdc_id", "description", "data_type"]
)

# SADECE TEMEL BESİNLER
food_df = food_df[
    food_df["data_type"].isin([
        "foundation_food",
        "sr_legacy_food"
    ])
].drop_duplicates(subset=["fdc_id"])

# =========================
# 2️⃣ food_nutrient
# =========================
food_nutrient_df = pd.read_csv(
    "/content/drive/MyDrive/datas/food_nutrient.csv",
    usecols=["fdc_id", "nutrient_id", "amount"],
    low_memory=False
)

# =========================
# 3️⃣ nutrient
# =========================
nutrient_df = pd.read_csv(
    "/content/drive/MyDrive/datas/nutrient.csv",
    usecols=["id", "name", "unit_name"]
)

# =========================
# USDA GERÇEK nutrient isimleri
# =========================
TARGET_NUTRIENTS = [
    "Energy",
    "Protein",
    "Total lipid (fat)",
    "Carbohydrate, by difference",
    "Sugars, total including NLEA",
    "Fiber, total dietary",
    "Sodium, Na"
]

nutrient_filtered = nutrient_df[
    nutrient_df["name"].isin(TARGET_NUTRIENTS)
].rename(columns={"id": "nutrient_id"}).drop_duplicates(subset=["nutrient_id"])

# =========================
# 4️⃣ food_nutrient JOIN nutrient
# =========================
food_nutrient_joined = food_nutrient_df.merge(
    nutrient_filtered,
    on="nutrient_id",
    how="inner"
)

# =========================
# 5️⃣ JOIN food (fdc_id filtresi burada KRİTİK)
# =========================
food_full = food_nutrient_joined.merge(
    food_df[["fdc_id", "description"]],
    on="fdc_id",
    how="inner"
)

# =========================
# 6️⃣ Temizleme
# =========================
food_full = food_full.dropna(subset=["amount"])
food_full["amount"] = pd.to_numeric(food_full["amount"], errors="coerce")
food_full = food_full.dropna(subset=["amount"])

food_full["food_name"] = (
    food_full["description"]
    .str.lower()
    .str.strip()
)

# =========================
# 7️⃣ Kontrol çıktıları
# =========================
print("🔍 Ara tablo (food_full):", food_full.shape)
print("🍎 Food unique (fdc_id):", food_df["fdc_id"].nunique())
print("🧪 Nutrient sayısı:", nutrient_filtered["nutrient_id"].nunique())


🔍 Ara tablo (food_full): (55883, 7)
🍎 Food unique (fdc_id): 8204
🧪 Nutrient sayısı: 7


In [ ]:
# =========================
# Energy → kcal normalize
# =========================
energy_df = food_full[
    food_full["name"] == "Energy"
].copy()

# kJ → kcal dönüşümü
energy_df.loc[
    energy_df["unit_name"] == "kJ", "amount"
] = energy_df["amount"] * 0.239006

foods_df = energy_df[[
    "fdc_id",
    "food_name",
    "amount"
]].rename(columns={
    "amount": "kcal_per_100g"
}).drop_duplicates(subset=["fdc_id"])

# =========================
# Nutrient node
# =========================
nutrients_df = nutrient_filtered[[
    "nutrient_id",
    "name",
    "unit_name"
]].drop_duplicates()

# =========================
# CONTAINS ilişkileri (Energy hariç)
# =========================
relations_df = food_full[
    food_full["name"] != "Energy"
][[
    "fdc_id",
    "name",
    "unit_name",
    "amount"
]].rename(columns={
    "name": "nutrient",
    "amount": "amount_per_100g"
})

print("✅ Normalize tamamlandı")
print(f"Food: {len(foods_df)}")
print(f"Nutrient: {len(nutrients_df)}")
print(f"Relations: {len(relations_df)}")


✅ Normalize tamamlandı
Food: 7928
Nutrient: 7
Relations: 40027


In [ ]:
!pip install neo4j

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 7.1 MB/s eta 0:00:00


In [ ]:
from neo4j import GraphDatabase

# =========================
# Neo4j bağlantı bilgileri
# =========================
URI = "bolt://13.61.1.136:7687"
AUTH = ("neo4j", "neo4j_sifre.")

driver = GraphDatabase.driver(URI, auth=AUTH)

def run(query):
    with driver.session() as session:
        session.run(query)

# =========================
# String güvenliği (ÇOK KRİTİK)
# =========================
def safe_str(text):
    if text is None:
        return ""
    return (
        str(text)
        .replace("\\", "\\\\")
        .replace('"', '\\"')
        .replace("\n", " ")
        .strip()
    )

# =========================
# Food node'ları
# =========================
for _, row in foods_df.iterrows():
    run(f'''
    MERGE (:Food {{
        fdc_id: {int(row["fdc_id"])},
        name: "{safe_str(row["food_name"])}",
        kcal_per_100g: {round(float(row["kcal_per_100g"]), 2)}
    }})
    ''')

# =========================
# Nutrient node'ları
# =========================
for _, row in nutrients_df.iterrows():
    run(f'''
    MERGE (:Nutrient {{
        nutrient_id: {int(row["nutrient_id"])},
        name: "{safe_str(row["name"])}",
        unit: "{safe_str(row["unit_name"])}"
    }})
    ''')

# =========================
# CONTAINS ilişkileri
# =========================
for _, row in relations_df.iterrows():
    run(f'''
    MATCH (f:Food {{fdc_id: {int(row["fdc_id"])}}})
    MATCH (n:Nutrient {{name: "{safe_str(row["nutrient"])}"}})
    MERGE (f)-[:CONTAINS {{
        amount_per_100g: {round(float(row["amount_per_100g"]), 2)},
        unit: "{safe_str(row["unit_name"])}"
    }}]->(n)
    ''')

print("🚀 Neo4j Knowledge Graph başarıyla oluşturuldu")


🚀 Neo4j Knowledge Graph başarıyla oluşturuldu


In [ ]:
!pip install -q transformers neo4j accelerate pillow apoc

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 325.4/325.4 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.0/624.0 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 122.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 732.6/732.6 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 10.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless

In [ ]:
import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from neo4j import GraphDatabase
from collections import Counter

# =========================================
# 1️⃣ Qwen2-VL Model
# =========================================
MODEL_ID = "Qwen/Qwen2-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()

# =========================================
# 2️⃣ Neo4j bağlantı
# =========================================
URI = "bolt://13.61.1.136:7687"
AUTH = ("neo4j", "neo4j_sifre.")

driver = GraphDatabase.driver(URI, auth=AUTH)

def neo4j_query(query, params=None):
    with driver.session() as session:
        result = session.run(query, params or {})
        return [r.data() for r in result]

# =========================================
# 3️⃣ Görseli 4x4 Grid'e Böl
# =========================================
def split_4x4(image: Image.Image):
    w, h = image.size
    patches = []
    for i in range(4):
        for j in range(4):
            patch = image.crop((
                j * w // 4,
                i * h // 4,
                (j + 1) * w // 4,
                (i + 1) * h // 4
            ))
            patches.append(patch)
    return patches

# =========================================
# 4️⃣ Qwen2-VL DOĞRU Vision Prompt
# =========================================
def predict_food(patch: Image.Image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": patch},
                {
                    "type": "text",
                    "text": (
                        "Identify the main food item in the image. "
                        "Use a simple generic food name. "
                        "No brand names. "
                        "Return ONLY the food name."
                    )
                }
            ]
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[text],
        images=[patch],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False
        )

    decoded = processor.decode(output[0], skip_special_tokens=True)

    # 🔥 KRİTİK SATIR
    answer = decoded.split("assistant")[-1].strip()

    return answer.lower()


# =========================================
# 5️⃣ Normalize / Synonyms
# =========================================
SYNONYMS = {
    "fried chicken": "chicken",
    "grilled chicken": "chicken",
    "white rice": "rice",
    "steamed rice": "rice",
    "hamburger": "burger",
    "cheeseburger": "burger"
}

def normalize_food(name):
    return SYNONYMS.get(name, name)

# =========================================
# 6️⃣ GraphRAG Query
# =========================================
GRAPH_RAG_QUERY = """
MATCH (f:Food)
WHERE f.name CONTAINS $food_name

OPTIONAL MATCH (f)-[:GOOD_FOR]->(g:Goal)
OPTIONAL MATCH (f)-[:BAD_FOR]->(d:Disease)
OPTIONAL MATCH (f)-[c:CONTAINS]->(n:Nutrient)

WHERE n.name IN [
  "Protein",
  "Fiber, total dietary",
  "Sugars, total including NLEA",
  "Sodium, Na",
  "Energy"
]

RETURN
  f.name AS food,
  f.kcal_per_100g AS calories,
  collect(DISTINCT g.name) AS good_for,
  collect(DISTINCT d.name) AS bad_for,
  collect({
    nutrient: n.name,
    amount: c.amount_per_100g,
    unit: n.unit
  }) AS nutrients
LIMIT 1
"""

# =========================================
# 7️⃣ FULL PIPELINE
# =========================================
def analyze_food_image(image_path):
    image = Image.open(image_path).convert("RGB")
    patches = split_4x4(image)

    predictions = []

    for p in patches:
        try:
            food = predict_food(p)
            food = normalize_food(food)
            if food:
                predictions.append(food)
        except Exception as e:
            print("⚠️ Patch error:", e)

    if not predictions:
        return {
            "detected_food": None,
            "graph_context": [],
            "error": "Vision model could not detect food",
            "raw_predictions": predictions
        }

    final_food = Counter(predictions).most_common(1)[0][0]

    graph_data = neo4j_query(
        GRAPH_RAG_QUERY,
        {"food_name": final_food}
    )

    return {
        "detected_food": final_food,
        "graph_context": graph_data,
        "raw_predictions": predictions
    }

# =========================================
# 8️⃣ TEST
# =========================================
result = analyze_food_image("/content/drive/MyDrive/datas/bigmac.webp")

print("🍎 Detected food:", result["detected_food"])
print("🧠 Graph context:")
print(result["graph_context"])


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

🍎 Detected food: burger
🧠 Graph context:
[{'food': 'interstate brands corp, wonder hamburger rolls', 'calories': 273.0, 'good_for': [], 'bad_for': ['Hypertension'], 'nutrients': [{'amount': 488.0, 'unit': 'MG', 'nutrient': 'Sodium, Na'}, {'amount': 2.6, 'unit': 'G', 'nutrient': 'Fiber, total dietary'}, {'amount': 8.07, 'unit': 'G', 'nutrient': 'Protein'}]}]


In [6]:
import os
import pyarrow.parquet as pq

# Senin tanımladığın path
DATA_DIR = "/content/drive/MyDrive/datas/food_nutrients_raw"

parquet_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".parquet")
]

print(f"Toplam {len(parquet_files)} dosya kontrol ediliyor...\n")

valid_files = []
corrupt_files = []

for f in parquet_files:
    filename = os.path.basename(f)
    try:
        # 1. Dosya boyutu kontrolü
        size = os.path.getsize(f)
        if size == 0:
            print(f"❌ BOŞ DOSYA: {filename} (0 bytes)")
            corrupt_files.append(f)
            continue

        # 2. Magic Bytes kontrolü (Parquet dosyaları 'PAR1' ile biter)
        with open(f, 'rb') as file_obj:
            file_obj.seek(-4, 2)  # Sondan 4 byte git
            footer = file_obj.read()
            if footer != b'PAR1':
                print(f"❌ MAGIC BYTE HATASI: {filename} (Footer: {footer})")
                corrupt_files.append(f)
                continue

        # 3. PyArrow ile okuma denemesi (Schema okuma)
        pq.read_schema(f)
        print(f"✅ SAĞLAM: {filename}")
        valid_files.append(f)

    except Exception as e:
        print(f"❌ OKUMA HATASI: {filename} -> {e}")
        corrupt_files.append(f)

print("-" * 30)
print(f"Sonuç: {len(valid_files)} sağlam, {len(corrupt_files)} bozuk dosya.")

Toplam 3 dosya kontrol ediliyor...

✅ SAĞLAM: 0002.parquet
✅ SAĞLAM: 0001.parquet
✅ SAĞLAM: 0000.parquet
------------------------------
Sonuç: 3 sağlam, 0 bozuk dosya.


In [8]:
# ===============================
# 0️⃣ GEREKLİ KÜTÜPHANELER
# ===============================
!pip install -q datasets pyarrow torch

# ===============================
# 1️⃣ IMPORTLAR
# ===============================
import os
from datasets import load_dataset
import torch

print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

# ===============================
# 2️⃣ DATASET PATH
# ===============================
DATA_DIR = "/content/drive/MyDrive/datas/food_nutrients_raw"

# parquet dosyalarını bul
parquet_files = [
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.endswith(".parquet")
]

print("Bulunan parquet dosyası:", len(parquet_files))
for f in parquet_files[:3]:
    print(" -", os.path.basename(f))

if len(parquet_files) == 0:
    raise RuntimeError("❌ parquet dosyası bulunamadı")

# ===============================
# 3️⃣ HF DATASET OLARAK YÜKLE
# ===============================
dataset = load_dataset(
    "parquet",
    data_files=parquet_files,
    split="train"
)

print("✅ Dataset yüklendi")
print(dataset)

# ===============================
# 4️⃣ SCHEMA KONTROLÜ
# ===============================
print("\n📌 Dataset kolonları:")
print(dataset.column_names)

print("\n📌 İlk örnek:")
print(dataset[0])

# ===============================
# 5️⃣ GÖRSEL + LABEL VAR MI?
# ===============================
image_cols = [c for c in dataset.column_names if "image" in c.lower()]
label_cols = [c for c in dataset.column_names if c not in image_cols]

print("\n🖼️ Olası görsel kolonları:", image_cols)
print("🏷️ Olası label kolonları:", label_cols)


Device: cuda
Bulunan parquet dosyası: 3
 - 0002.parquet
 - 0001.parquet
 - 0000.parquet
✅ Dataset yüklendi
Dataset({
    features: ['image', 'id', 'split', 'ingredients', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein'],
    num_rows: 3260
})

📌 Dataset kolonları:
['image', 'id', 'split', 'ingredients', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein']

📌 İlk örnek:
{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGB size=640x480 at 0x7AFA50C35C10>, 'id': 'dish_1565021054', 'split': 'test', 'ingredients': [{'id': 'ingr_0000000050', 'name': 'tofu', 'grams': 71.0, 'calories': 51.83, 'fat': 3.408, 'carb': 1.349, 'protein': 5.68}, {'id': 'ingr_0000000337', 'name': 'waffles', 'grams': 73.0, 'calories': 211.919, 'fat': 10.439, 'carb': 23.871, 'protein': 5.767}], 'total_calories': 263.749023, 'total_mass': 144.0, 'total_fat': 13.847, 'total_carb': 25.220001, 'total_protein': 11.447001}

🖼️ Olası görsel kolonları: ['image']
🏷️ Olası la

In [17]:
# =====================================================
# 0️⃣ KURULUM
# =====================================================
!pip install -q transformers accelerate datasets peft pillow bitsandbytes

# =====================================================
# 1️⃣ IMPORT
# =====================================================
import torch
from datasets import load_dataset
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =====================================================
# 2️⃣ DATASET (PARQUET'TEN)
# =====================================================
DATA_DIR = "/content/drive/MyDrive/datas/food_nutrients_raw"

dataset = load_dataset(
    "parquet",
    data_files=f"{DATA_DIR}/*.parquet",
    split="train"
)

dataset = dataset.filter(lambda x: x["split"] == "test")
print("Toplam train örneği:", len(dataset))
print("Kolonlar:", dataset.column_names)

# =====================================================
# 3️⃣ MODEL & PROCESSOR
# =====================================================
MODEL_NAME = "Qwen/Qwen2-VL-7B-Instruct"

processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    load_in_4bit=True
)

# =====================================================
# 4️⃣ LoRA (A100 VAR AMA YİNE DE GEREKLİ)
# =====================================================
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# =====================================================
# 5️⃣ DATA COLLATOR
# =====================================================
def data_collator(batch):
    conversations = []
    images = []

    for x in batch:
        conversations.append([
            {
                "role": "user",
                "content": [
                    {"type": "image"},
                    {
                        "type": "text",
                        "text": "Bu yemeğin toplam kalorisini tahmin et. Cevabı sadece sayı olarak ver."
                    }
                ]
            },
            {
                "role": "assistant",
                "content": [
                    {
                        "type": "text",
                        "text": str(round(x["total_calories"], 2))
                    }
                ]
            }
        ])
        images.append(x["image"])

    # 🔥 QWEN2-VL CHAT TEMPLATE
    text_inputs = processor.apply_chat_template(
        conversations,
        tokenize=False,
        add_generation_prompt=False
    )

    model_inputs = processor(
        text=text_inputs,
        images=images,
        padding=True,
        return_tensors="pt"
    )

    model_inputs["labels"] = model_inputs["input_ids"].clone()
    return model_inputs
# =====================================================
# 6️⃣ TRAINING
# =====================================================
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/qwen2_vl_food_calorie",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=3,
    fp16=True,
    logging_steps=32,
    save_strategy="epoch",
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    data_collator=data_collator
)

trainer.train()

# =====================================================
# 7️⃣ KAYDET
# =====================================================
trainer.save_model("/content/drive/MyDrive/qwen2_vl_food_calorie")
processor.save_pretrained("/content/drive/MyDrive/qwen2_vl_food_calorie")

print("✅ Vision fine-tune tamamlandı")


Device: cuda
Toplam train örneği: 3260
Kolonlar: ['image', 'id', 'split', 'ingredients', 'total_calories', 'total_mass', 'total_fat', 'total_carb', 'total_protein']


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

trainable params: 10,092,544 || all params: 8,301,468,160 || trainable%: 0.1216


Step,Training Loss
32,9.651100
64,7.090000
96,7.017700
128,7.005800
160,7.001200
192,6.999000
224,6.998300
256,6.996500
288,6.996700
320,6.993900


✅ Vision fine-tune tamamlandı


In [29]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
import torch
from PIL import Image

model_path = "/content/drive/MyDrive/qwen2_vl_food_calorie"

processor = AutoProcessor.from_pretrained(model_path)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto"
)

image = Image.open("/content/drive/MyDrive/datas/badem.jpeg")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {
                "type": "text",
                "text": (
                    "Bu yemeği görselden analiz et.\n\n"
                    "Cevabı SADECE aşağıdaki JSON formatında ver:\n"
                    "{\n"
                    '  "estimated_calories": number,\n'
                    '  "reason": "short visual explanation"\n'
                    "}"
                )
            }
        ]
    }
]

# Qwen2-VL için doğru chat template
prompt_text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(
    text=prompt_text,
    images=image,
    return_tensors="pt"
).to("cuda")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=False,
        temperature=0.0,
        repetition_penalty=1.2,
    )

response = processor.decode(
    output_ids[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
)

print(response)


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

{
  "estimated_calories": 231.68,
  "reason": "The image shows a bowl of nuts and seeds which contain high amount of calories."
}


In [31]:

# ---------- INSTALL ----------
!pip install -q transformers accelerate datasets peft pillow neo4j bitsandbytes

# ---------- IMPORT ----------
import torch
from neo4j import GraphDatabase
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from PIL import Image

# ---------- DEVICE ----------
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# =====================================================
# 1️⃣ NEO4J CONNECTION
# =====================================================
NEO4J_URI = "bolt://13.61.1.136:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "neo4j_sifre."

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USER, NEO4J_PASS)
)

def retrieve_from_neo4j(question, limit=3):
    query = """
    MATCH (n)
    WHERE any(k IN keys(n) WHERE toLower(toString(n[k])) CONTAINS toLower($q))
    RETURN n
    LIMIT $limit
    """

    docs = []
    with driver.session() as session:
        results = session.run(query, q=question, limit=limit)
        for r in results:
            node = dict(r["n"])
            text = " | ".join([f"{k}: {v}" for k, v in node.items()])
            docs.append(text)

    return "\n".join(docs)

# =====================================================
# 2️⃣ MODEL & PROCESSOR
# =====================================================
MODEL_PATH = "/content/drive/MyDrive/qwen2_vl_food_calorie"

processor = AutoProcessor.from_pretrained(MODEL_PATH)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

# =====================================================
# 3️⃣ CHATBOT (RAG)
# =====================================================
def ask_chatbot(question):
    context = retrieve_from_neo4j(question)

    messages = [
        {
            "role": "system",
            "content": (
                "Sen profesyonel bir diyetisyen yapay zekasın.\n"
                "SADECE aşağıdaki Neo4j bilgisini kullan.\n"
                "Eğer bilgi yoksa: 'Bu konuda yeterli bilgi yok' de.\n\n"
                f"NEO4J_BİLGİ:\n{context if context else 'YOK'}"
            )
        },
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=prompt,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.0,
            do_sample=False
        )

    return processor.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

# =====================================================
# 4️⃣ IMAGE + RAG CHATBOT
# =====================================================
def ask_with_image(question, image_path):
    image = Image.open(image_path)

    context = retrieve_from_neo4j(question)

    messages = [
        {
            "role": "system",
            "content": (
                "Sen profesyonel bir diyetisyen yapay zekasın.\n"
                "Görsel ve Neo4j bilgisini birlikte kullan.\n"
                "Bilgi yoksa açıkça söyle.\n\n"
                f"NEO4J_BİLGİ:\n{context if context else 'YOK'}"
            )
        },
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": question}
            ]
        }
    ]

    prompt = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=prompt,
        images=image,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=128
        )

    return processor.decode(
        out[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

# =====================================================
# 5️⃣ TEST
# =====================================================
print("---- TEXT RAG TEST ----")
print(
    ask_chatbot(
        "Kilo vermek isteyen biri için günlük karbonhidrat miktarı ne olmalı?"
    )
)

print("\n---- IMAGE + RAG TEST ----")
print(
    ask_with_image(
        "Bu besin sporcular için uygun mu?",
        "/content/drive/MyDrive/datas/badem.jpeg"
    )
)


Device: cuda


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

---- TEXT RAG TEST ----
Günlük karbonhidrat tüketimi, kilo verme hedefine bağlı olarak değişir. Genel olarak, bir kişinin günlük karbonhidrat tüketimi, toplam kalori gereksinimlerinin %40 ila %60'ına kadar olmalıdır. Ancak, bu oranlar, kişinin yaş, cinsiyeti, egzersiz seviyesi ve sağlık durumuna bağlı olarak değişebilir. Ayrıca, kilo verme hedefine bağlı olarak da değişebilir. Örneğin, bir kişinin günlük 2000 kalori gereksinimine göre, günlük karbonhidrat tüketimi 800-1200 gr arasında olmalıdır. Ancak, bu değerler genel bir öneridir ve herkes için farklı olabilir. Bu nedenle, bir kişinin günlük karbonhidrat tüketimini belirlemek için, profesyonel bir diyetisyenin tavsiyesi önemlidir.

---- IMAGE + RAG TEST ----
Evet, bu besin sporcular için uygun. Bu nedenle, sporcuların enerji gereksinimlerini karşılamak için bu besinleri tüketebilirler.
